# TPWRS PRP Publications by Year: Six Affinity Bands

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
benchmark_root = repo_root / "data" / "results" / "affinity_benchmark"
mapping_path = repo_root / "data" / "processed" / "ra_grouping_ra2025.csv"
band_order = ["Low-", "Low+", "Medium-", "Medium+", "High-", "High+"]
programme_order = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]


def load_programme_mapping():
    programme_mapping = pd.read_csv(mapping_path)[["RA2025", "Grouping"]]
    return (
        programme_mapping
        .dropna()
        .drop_duplicates()
        .rename(columns={"RA2025": "RA2025_ID", "Grouping": "PRP_Name"})
    )


def assign_programme_bands(ra_adjudication_df):
    programme_mapping = load_programme_mapping()
    question_scores = ra_adjudication_df.copy()
    question_scores["RA2025_ID"] = pd.to_numeric(question_scores["RA2025_ID"], errors="coerce")
    question_scores["Final_Affinity"] = pd.to_numeric(question_scores["Final_Affinity"], errors="coerce")
    programme_mapping["RA2025_ID"] = pd.to_numeric(programme_mapping["RA2025_ID"], errors="coerce")
    question_scores = question_scores.merge(programme_mapping, on="RA2025_ID", how="left")
    question_scores = question_scores.dropna(
        subset=["Abstract_Index", "PRP_Name", "RA2025_ID", "Final_Affinity"]
    )
    programme_scores = (
        question_scores
        .sort_values(
            ["Abstract_Index", "PRP_Name", "Final_Affinity", "RA2025_ID"],
            ascending=[True, True, False, True],
        )
        .drop_duplicates(["Abstract_Index", "PRP_Name"], keep="first")
        .rename(
            columns={
                "RA2025_ID": "Highest_Question_ID",
                "RA_Question": "Highest_Question_Text",
                "Final_Affinity": "Highest_Question_Affinity",
            }
        )
    )
    programme_scores["Affinity_Band"] = pd.cut(
        programme_scores["Highest_Question_Affinity"],
        bins=[0, 20, 40, 60, 80, 100, float('inf')],
        labels=band_order,
        right=False,
    )
    return programme_scores


assignments_df = pd.concat(
    [
        assign_programme_bands(pd.read_csv(path)).assign(
            Year=int(path.parent.name.split("_")[1])
        )
        for path in sorted(benchmark_root.glob("tpwrs_*/adjudicated_ra_affinities.csv"))
    ],
    ignore_index=True,
 )

papers_per_year = assignments_df.groupby("Year")["Abstract_Index"].nunique()
coverage_bands = (
    assignments_df
    .groupby(["Year", "PRP_Name", "Affinity_Band"], observed=True)["Abstract_Index"]
    .nunique()
    .rename("Papers")
    .reindex(
        pd.MultiIndex.from_product(
            [papers_per_year.index, programme_order, band_order],
            names=["Year", "PRP_Name", "Affinity_Band"],
        ),
        fill_value=0,
    )
    .reset_index()
    .merge(papers_per_year.rename("Papers_Total"), on="Year")
 )
coverage_bands["Share"] = 100 * coverage_bands["Papers"] / coverage_bands["Papers_Total"]
coverage_bands["Affinity_Band"] = pd.Categorical(
    coverage_bands["Affinity_Band"], categories=band_order, ordered=True
 )


figure = px.area(
    coverage_bands,
    x="Year",
    y="Share",
    color="Affinity_Band",
    facet_col="PRP_Name",
    facet_col_wrap=2,
    category_orders={
        "PRP_Name": programme_order,
        "Affinity_Band": band_order[::-1],
    },
 )
# figure.update_yaxes(range=[0, 80])
figure.show()

In [2]:
group_highest_scores = (
    assignments_df
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)["Highest_Question_Affinity"]
    .transform("max")
)

highest_scoring_indices_by_prp_band = (
    assignments_df
    .loc[assignments_df["Highest_Question_Affinity"].eq(group_highest_scores)]
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)
    .agg(
        Highest_Score=("Highest_Question_Affinity", "first"),
        Year_Abstract_Question_IDs=(
            "Highest_Question_ID",
            lambda question_ids: list(
                zip(
                    assignments_df.loc[question_ids.index, "Year"],
                    assignments_df.loc[question_ids.index, "Abstract_Index"],
                    question_ids,
                    strict=True,
                )
            ),
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [programme_order, band_order], names=["PRP_Name", "Affinity_Band"]
        )
    )
)
highest_scoring_indices_by_prp_band

Highest_Score  \
PRP_Name        Affinity_Band                  
IBR             Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.6   
                High-                   97.6   
                High+                    NaN   
Stability       Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.8   
                High-                   99.4   
                High+                  105.8   
DER             Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.8   
                High-                   98.2   
                High+                  102.0   
CROF            Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.8   
                High-                   99.6   
                High+                  109.6   
System Services Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.8   
                High-                   99.6   
                High+                  105.6   
Planning        Low-                    19.8   
                Low+                    39.8   
                Medium-                 59.8   
                Medium+                 79.8   
                High-                   99.8   
                High+                  105.2   

                                                      Year_Abstract_Question_IDs  
PRP_Name        Affinity_Band                                                     
IBR             Low-           [(2015, 10.1109/TPWRS.2014.2359234, 1), (2015,...  
                Low+           [(2017, 10.1109/TPWRS.2016.2615075, 4), (2020,...  
                Medium-        [(2019, 10.1109/TPWRS.2019.2928266, 4), (2025,...  
                Medium+        [(2018, 10.1109/TPWRS.2017.2713981, 1), (2018,...  
                High-                    [(2020, 10.1109/TPWRS.2020.2968114, 2)]  
                High+                                                        NaN  
Stability       Low-           [(2015, 10.1109/TPWRS.2014.2376494, 9), (2016,...  
                Low+           [(2016, 10.1109/TPWRS.2015.2434938, 9), (2016,...  
                Medium-                  [(2018, 10.1109/TPWRS.2017.2776741, 7)]  
                Medium+        [(2019, 10.1109/TPWRS.2019.2915249, 7), (2023,...  
                High-                    [(2022, 10.1109/TPWRS.2021.3126809, 7)]  
                High+                    [(2026, 10.1109/TPWRS.2025.3616391, 8)]  
DER             Low-           [(2015, 10.1109/TPWRS.2014.2322258, 12), (2015...  
                Low+           [(2018, 10.1109/TPWRS.2017.2696571, 12), (2019...  
                Medium-        [(2015, 10.1109/TPWRS.2014.2359457, 18), (2016...  
                Medium+        [(2017, 10.1109/TPWRS.2017.2652485, 18), (2018...  
                High-                   [(2023, 10.1109/TPWRS.2022.3196708, 12)]  
                High+          [(2021, 10.1109/TPWRS.2021.3052958, 18), (2026...  
CROF            Low-           [(2019, 10.1109/TPWRS.2019.2903552, 25), (2021...  
                Low+           [(2015, 10.1109/TPWRS.2014.2347373, 20), (2015...  
                Medium-        [(2015, 10.1109/TPWRS.2014.2338735, 21), (2016...  
                Medium+        [(2019, 10.1109/TPWRS.2018.2856283, 21), (2019...  
                High-                   [(2023, 10.1109/TPWRS.2022.3211452, 20)]  
                High+                   [(2026, 10.1109/TPWRS.2026.3690326, 25)]  
System Services Low-           [(2015, 10.1109/T

In [3]:
coverage_bands.groupby(["PRP_Name", "Affinity_Band"])["Papers"].sum()

PRP_Name         Affinity_Band
CROF             Low-              398
                 Low+             1952
                 Medium-          1986
                 Medium+          1328
                 High-             331
                 High+              10
DER              Low-             2713
                 Low+             2003
                 Medium-           751
                 Medium+           417
                 High-             118
                 High+               3
IBR              Low-             3247
                 Low+             1278
                 Medium-           720
                 Medium+           622
                 High-             138
                 High+               0
Planning         Low-             1864
                 Low+             1976
                 Medium-          1107
                 Medium+           788
                 High-             269
                 High+               1
Stability        Low-            

In [4]:
px.histogram(
    assignments_df,
    x= "Affinity_Band",
    # color = "Year"
    facet_col = "PRP_Name",
    facet_col_wrap=3,
    category_orders={ "Affinity_Band": band_order[::-1]}
)

In [5]:
assignments_df

,Abstract_Index,Document Title,Highest_Question_ID,Highest_Question_Text,N_Models,Affinity_Mean,Affinity_Std,Affinity_Min,Affinity_Max,Affinity_Range,...,agreement_max,Highest_Question_Affinity,Final_Reason,Final_Method,Model_Slug,Cosine_x100_Mean,Run_ID,PRP_Name,Affinity_Band,Year
0,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,27,how can risk-based methods (e.g. probabilistic...,5,31.4,14.099645,14.0,50.0,36.0,...,0.6,31.4,NaN,ensemble_mean,NaN,15.9306,tpwrs_2015_tpwrs_2015,CROF,Low+,2015
1,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,12,what mechanisms and appropriate aggregate der ...,5,15.8,12.774976,0.0,30.0,30.0,...,1.0,15.8,NaN,ensemble_mean,NaN,15.2541,tpwrs_2015_tpwrs_2015,DER,Low-,2015
2,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,1,what is the future of active power balance as ...,5,15.0,7.071068,5.0,20.0,15.0,...,1.0,15.0,NaN,ensemble_mean,NaN,33.8061,tpwrs_2015_tpwrs_2015,IBR,Low-,2015
3,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,41,what additional probabilistic planning methods...,5,53.6,22.634045,25.0,85.0,60.0,...,0.6,53.6,NaN,ensemble_mean,NaN,43.4438,tpwrs_2015_tpwrs_2015,Planning,Medium-,2015
4,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,9,what analytical tools and models should be pro...,5,61.4,15.485477,35.0,72.0,37.0,...,0.8,61.4,NaN,ensemble_mean,NaN,40.0692,tpwrs_2015_tpwrs_2015,Stability,Medium+,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36025,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,15,"what aggregate quantities must be monitored, s...",5,24.8,9.757049,10.0,34.0,24.0,...,1.0,24.8,NaN,ensemble_mean,NaN,36.4899,tpwrs_2026_tpwrs_2026,DER,Low+,2026
36026,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,1,what is the future of active power balance as ...,5,75.8,10.871982,60.0,90.0,30.0,...,0.6,75.8,NaN,ensemble_mean,NaN,39.2350,tpwrs_2026_tpwrs_2026,IBR,Medium+,2026
36027,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,37,what studies and metrics are required to ident...,5,25.6,9.316652,18.0,40.0,22.0,...,0.8,25.6,NaN,ensemble_mean,NaN,33.3348,tpwrs_2026_tpwrs_2026,Planning,Low+,2026
36028,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,9,what analytical tools and models should be pro...,5,75.6,6.268971,70.0,85.0,15.0,...,0.8,75.6,NaN,ensemble_mean,NaN,51.4229,tpwrs_2026_tpwrs_2026,Stability,Medium+,2026


In [6]:
paper_programme_bands = assignments_df.pivot(
    index=["Year", "Abstract_Index"],
    columns="PRP_Name",
    values="Affinity_Band",
)
target_band_order = band_order

source_target_band_summaries = []
for source_prp in programme_order:
    source_prp_is_high = paper_programme_bands[source_prp].isin(["High-", "High+"])

    for target_prp in programme_order:
        if target_prp == source_prp:
            continue

        for affinity_band in target_band_order:
            target_prp_in_band = paper_programme_bands[target_prp].eq(affinity_band)
            papers_with_high_source_and_target_band = source_prp_is_high & target_prp_in_band
            conditional_probability = papers_with_high_source_and_target_band[source_prp_is_high].mean()
            target_prp_baseline_probability = target_prp_in_band.mean()
            lift = (
                conditional_probability / target_prp_baseline_probability
                if target_prp_baseline_probability > 0
                else float("nan")
            )

            source_target_band_summaries.append(
                {
                    "Source_PRP": source_prp,
                    "Target_PRP": target_prp,
                    "Affinity_Band": affinity_band,
                    "Papers_with_High_Source_PRP": source_prp_is_high.sum(),
                    "Papers_with_High_Source_and_Target_PRP_Band": (
                        papers_with_high_source_and_target_band.sum()
                    ),
                    "Probability_of_Target_PRP_Band_given_High_Source_PRP": (
                        conditional_probability
                    ),
                    "Baseline_Probability_of_Target_PRP_Band": target_prp_baseline_probability,
                    "Lift_vs_Target_PRP_Baseline": lift,
                }
            )

source_target_band_diversification_summary = pd.DataFrame(source_target_band_summaries)
source_target_band_diversification_summary

,Source_PRP,Target_PRP,Affinity_Band,Papers_with_High_Source_PRP,Papers_with_High_Source_and_Target_PRP_Band,Probability_of_Target_PRP_Band_given_High_Source_PRP,Baseline_Probability_of_Target_PRP_Band,Lift_vs_Target_PRP_Baseline
0,IBR,Stability,Low-,138,0,0.000000,0.135054,0.000000
1,IBR,Stability,Low+,138,6,0.043478,0.361199,0.120372
2,IBR,Stability,Medium-,138,30,0.217391,0.284263,0.764754
3,IBR,Stability,Medium+,138,84,0.608696,0.165862,3.669897
4,IBR,Stability,High-,138,14,0.101449,0.051957,1.952573
...,...,...,...,...,...,...,...,...
175,Planning,System Services,Low+,270,168,0.622222,0.422315,1.473361
176,Planning,System Services,Medium-,270,46,0.170370,0.168193,1.012945
177,Planning,System Services,Medium+,270,28,0.103704,0.086595,1.197578
178,Planning,System Services,High-,270,7,0.025926,0.038968,0.665321


In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

source_prp_heatmaps = {}
for source_prp in programme_order:
    source_prp_heatmaps[source_prp] = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Probability_of_Target_PRP_Band_given_High_Source_PRP",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_heatmap = source_prp_heatmaps[source_prp]
    heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_heatmap,
            x=source_prp_heatmap.columns,
            y=source_prp_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.1%}",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Probability: %{z:.1%}<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

heatmap_figure.update_layout(
    title="Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#fcfcfb"],
            [0.5, "#86b6ef"],
            [0.75, "#2a78d6"],
            [1, "#104281"],
        ],
        "cmin": 0,
        "cmax": source_target_band_diversification_summary[
            "Probability_of_Target_PRP_Band_given_High_Source_PRP"
        ].max(),
        "colorbar": {"title": "Probability", "tickformat": ".0%"},
    },
)
heatmap_figure.update_xaxes(title_text="Target affinity band")
heatmap_figure.update_yaxes(title_text="Target PRP")
heatmap_figure.show()

In [8]:
lift_heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_lift_heatmap = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Lift_vs_Target_PRP_Baseline",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

    lift_heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_lift_heatmap,
            x=source_prp_lift_heatmap.columns,
            y=source_prp_lift_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.2f}x",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Lift: %{z:.2f}x<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

lift_heatmap_figure.update_layout(
    title="Lift of Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#e34948"],
            [0.5, "#fcfcfb"],
            [1, "#2a78d6"],
        ],
        "cmin": 0,
        "cmid": 1,
        "colorbar": {"title": "Lift vs. baseline", "tickformat": ".1f"},
    },
)
lift_heatmap_figure.update_xaxes(title_text="Target affinity band")
lift_heatmap_figure.update_yaxes(title_text="Target PRP")
lift_heatmap_figure.show()